<!--
Copyright (c) 2026 Milin Patel
Hochschule Kempten - University of Applied Sciences
Campustage 2026 - Automatisiertes Fahren! Sicher durch Zufall?
MIT License - see LICENSE.
-->

# Sicher durch Zufall? Wie viel Testen reicht?
### Vertiefung zum Workshop-Notebook *Der KI-Blick*

Hochschule Kempten, Institut für Fahrerassistenz und vernetzte Mobilität (IFM) · Milin Patel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/campustage-2026-sicher-durch-zufall/blob/main/01_Perception_Systems/notebooks/00_vertiefung_wie_viel_testen_reicht.ipynb)

---

### Worum es geht

Ein automatisiertes Fahrzeug muss in sehr vielen Situationen funktionieren. Wie lässt sich zeigen, dass es sicher genug ist?
Naheliegend ist, es in vielen zufällig ausgewählten Situationen zu testen. Dieses Notebook untersucht, was solche Tests aussagen und was nicht.

| Teil | Inhalt |
|---|---|
| 1 · Zufällige Testfälle | Das Modell in vielen zufälligen Sichtbedingungen prüfen |
| 2 · Was null Fehler bedeuten | Die Dreierregel |
| 3 · Systematisch suchen | Grenzfälle gezielt finden statt zufällig |
| Abschluss | Ergebnis und Fachbegriffe |

Dauer: etwa 10 Minuten. Es hilft, vorher das Workshop-Notebook *Der KI-Blick* bearbeitet zu haben.

### Bedienung

Im Menü **Laufzeit → Alle ausführen** wählen (Google-Konto nötig) und warten, bis unter *Vorbereitung* **Bereit** erscheint.
Danach wird alles über Schaltflächen und Regler bedient.

In [ ]:
# @title Vorbereitung: Modell laden
%pip install -q ultralytics==8.4.216 matplotlib
import io, time
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import ipywidgets as W
from IPython.display import display
from ultralytics import YOLO
from ultralytics.utils import ASSETS

try:
    from google.colab import output as _colab
    _colab.enable_custom_widget_manager()
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ---------- Farben, Texte ----------
ORANGE, BLAU, GRAU, LILA = (225, 120, 0), (40, 110, 200), (105, 105, 115), (120, 60, 170)
DE = {"person": "Person", "car": "Auto", "bus": "Bus", "truck": "LKW", "bicycle": "Fahrrad",
      "motorcycle": "Motorrad", "stop sign": "Stoppschild", "traffic light": "Ampel",
      "train": "Zug", "handbag": "Handtasche", "backpack": "Rucksack", "umbrella": "Schirm",
      "skateboard": "Skateboard", "dog": "Hund", "cat": "Katze", "bird": "Vogel",
      "tie": "Krawatte", "cell phone": "Handy", "bench": "Bank", "chair": "Stuhl",
      "fire hydrant": "Hydrant", "parking meter": "Parkuhr", "sports ball": "Ball",
      "suitcase": "Koffer", "bottle": "Flasche", "cup": "Becher", "laptop": "Laptop",
      "potted plant": "Pflanze", "tennis racket": "Tennisschläger", "baseball bat": "Schläger",
      "horse": "Pferd", "kite": "Drachen", "clock": "Uhr", "book": "Buch", "tv": "Bildschirm"}
def de(name): return DE.get(name, name)
def zahl(x):
    """Konfidenz oder Schwellenwert als deutsche Dezimalzahl, abgerundet (0,2497 wird 0,24, nicht 0,25)."""
    return f"{int(100 * x + 1e-6) / 100:.2f}".replace(".", ",")
def sek(x): return f"{x:.2f}".replace(".", ",") + " s"

def karte(text, art="info"):
    farben = {"info": ("#eef4fb", "#1d4e89"), "gut": ("#e9f6ee", "#1e6b3a"),
              "achtung": ("#fdf0e4", "#8a4b0f"), "frage": ("#f3eefa", "#5b2d8a")}
    bg, fg = farben[art]
    return W.HTML(f'<div style="background:{bg};color:{fg};border-left:6px solid {fg};'
                  f'padding:10px 14px;margin:6px 0;border-radius:6px;font-size:15px;'
                  f'line-height:1.5;font-family:sans-serif">{text}</div>')

def kacheln(werte):
    """werte: Liste von (Titel, Zahl, Farbe)."""
    html = "".join(
        f'<div style="flex:1;min-width:110px;background:{f};color:white;border-radius:8px;'
        f'padding:8px 10px;text-align:center;font-family:sans-serif">'
        f'<div style="font-size:13px">{t}</div><div style="font-size:30px;font-weight:bold">{z}</div></div>'
        for t, z, f in werte)
    return W.HTML(f'<div style="display:flex;gap:8px;flex-wrap:wrap;margin:6px 0">{html}</div>')

def knopf(text, stil="primary", breite="240px"):
    return W.Button(description=text, button_style=stil, layout=W.Layout(width=breite, height="38px"))

def sicher(ausgabe):
    """Fehler in Knopf-Funktionen freundlich anzeigen statt still zu verschlucken."""
    def deko(fn):
        def innen(*a, **k):
            try:
                return fn(*a, **k)
            except Exception as e:
                with ausgabe:
                    display(karte(f"Es ist ein Fehler aufgetreten ({type(e).__name__}: {e}). "
                                  "Bitte die Zelle erneut ausführen (▶ links).", "achtung"))
        return innen
    return deko

# ---------- Bilder ----------
def arbeitsbild(img):
    """Auf 640 Pixel (lange Seite) bringen, Seiten durch 32 teilbar."""
    img = img.convert("RGB")
    f = 640 / max(img.size)
    w = max(64, int(round(img.width * f / 32)) * 32)
    h = max(64, int(round(img.height * f / 32)) * 32)
    return img.resize((w, h), Image.LANCZOS)

def jpg(img):
    b = io.BytesIO(); img.save(b, "JPEG", quality=88); return b.getvalue()

def zeige(img, breite="520px"):
    return W.Image(value=jpg(img), format="jpeg", layout=W.Layout(width="100%", max_width=breite))

def nebeneinander(*teile):
    return W.HBox([W.VBox([W.HTML(f'<b style="font-family:sans-serif;font-size:15px">{t}</b>'),
                           zeige(b, "420px")], layout=W.Layout(width=f"{96 // len(teile)}%"))
                   for t, b in teile])

def _schrift(groesse):
    for n in ("DejaVuSans-Bold.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
              "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"):
        try: return ImageFont.truetype(n, groesse)
        except Exception: pass
    try: return ImageFont.load_default(size=groesse)
    except TypeError: return ImageFont.load_default()

def _gestrichelt(d, box, farbe, lw, l=9):
    x1, y1, x2, y2 = box
    for (ax, ay), (bx, by) in (((x1, y1), (x2, y1)), ((x2, y1), (x2, y2)), ((x2, y2), (x1, y2)), ((x1, y2), (x1, y1))):
        n = max(2, int(max(abs(bx - ax), abs(by - ay)) / l))
        for i in range(0, n, 2):
            a, b = i / n, min(1, (i + 1) / n)
            d.line([(ax + (bx - ax) * a, ay + (by - ay) * a), (ax + (bx - ax) * b, ay + (by - ay) * b)], fill=farbe, width=lw)

def _etikett(d, x, y, text, farbe, f):
    l, t, r, b = d.textbbox((0, 0), text, font=f)
    th = b - t
    x = max(0, min(x, d.im.size[0] - (r - l) - 8))
    y = y - th - 8 if y - th - 8 >= 0 else y + 2
    d.rectangle([x, y, x + (r - l) + 8, y + th + 6], fill=farbe)
    d.text((x + 4, y + 3 - t), text, fill=(255, 255, 255), font=f)

def zeichne(img, boxen, grenze, vermutungen=False, nummern=None, nur_menschen=False):
    """Boxen über der Grenze: durchgezogen. Darunter (wenn gewünscht): grau gestrichelt."""
    out = img.copy(); d = ImageDraw.Draw(out)
    f = _schrift(max(13, img.width // 34)); lw = max(2, img.width // 170)
    for b in sorted(boxen, key=lambda b: b["conf"]):
        if nur_menschen and b["name"] != "person":
            continue
        drin = b["conf"] >= grenze
        if not drin and not vermutungen:
            continue
        box = [int(v) for v in b["xyxy"]]
        if drin:
            farbe = ORANGE if b["name"] == "person" else BLAU
            d.rectangle(box, outline=farbe, width=lw)
        else:
            farbe = GRAU
            _gestrichelt(d, box, farbe, max(1, lw - 1))
        _etikett(d, box[0], box[1], f"{de(b['name'])} {zahl(b['conf'])}", farbe, f)
    if nummern:
        fg = _schrift(max(22, img.width // 14))
        for i, b in enumerate(nummern, 1):
            x1, y1, x2, y2 = b["xyxy"]
            cx, cy = (x1 + x2) / 2, y1 + (y2 - y1) * 0.15
            r = max(16, img.width // 26)
            d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=LILA, outline=(255, 255, 255), width=3)
            d.text((cx, cy), str(i), fill=(255, 255, 255), font=fg, anchor="mm")
    return out

# ---------- Modell ----------
MODELL = YOLO("yolov8n.pt")
NAMEN = MODELL.names
STANDARD = 0.30
_cache = {}

def erkenne_alles(img):
    """Einmal mit sehr niedrigem Schwellenwert rechnen; danach wird nur noch gefiltert (darum reagiert der Regler sofort)."""
    key = hash(img.tobytes())
    if key not in _cache:
        if len(_cache) > 60: _cache.clear()
        t = time.perf_counter()
        r = MODELL.predict(img, conf=0.05, imgsz=640, verbose=False)[0]
        dauer = time.perf_counter() - t
        boxen = [dict(name=NAMEN[int(b.cls[0])], conf=float(b.conf[0]), xyxy=[float(v) for v in b.xyxy[0]])
                 for b in r.boxes]
        _cache[key] = (boxen, dauer)
    return _cache[key]

def personen(boxen, grenze):
    return [b for b in boxen if b["name"] == "person" and b["conf"] >= grenze]

def iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    iw, ih = max(0, min(ax2, bx2) - max(ax1, bx1)), max(0, min(ay2, by2) - max(ay1, by1))
    i = iw * ih
    return i / ((ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - i + 1e-9)

def bester_treffer(boxen, ziel):
    """Höchste Konfidenz einer Person an der Stelle des Ziels (0, wenn dort keine mehr erkannt wird)."""
    werte = [b["conf"] for b in boxen if b["name"] == "person" and iou(b["xyxy"], ziel["xyxy"]) > 0.3]
    return max(werte, default=0.0)

S = dict(bild=None, quelle=None, gezaehlt=None, zeit_mensch=None, zeit_modell=None, modell_personen=None,
         grenze=None, zielkonflikt=None, wetter_tipp=None, wetter_ergebnis=None, nebel_entscheidung=None,
         angriff=None, antwort_vorher=None, antwort_nachher=None)

def grenze(): return S["grenze"] if S["grenze"] is not None else STANDARD

def neues_bild(img, quelle):
    S.update(bild=arbeitsbild(img), quelle=quelle, gezaehlt=None, zeit_mensch=None, zeit_modell=None,
             modell_personen=None, wetter_ergebnis=None, angriff=None)

# ---------- Wetter (simuliert) ----------
def wetter(img, art, staerke):
    """staerke von 0 bis 1."""
    if staerke <= 0 or art == "Klar":
        return img
    rng = np.random.default_rng(7)
    if art == "Nebel":
        grau = Image.new("RGB", img.size, (208, 208, 212))
        return Image.blend(img, grau, 0.88 * staerke).filter(ImageFilter.GaussianBlur(0.5 + 2.5 * staerke))
    if art == "Regen":
        o = img.copy(); d = ImageDraw.Draw(o); w, h = img.size
        lang = max(8, h // 30)
        for _ in range(int(w * h / 150 * staerke)):
            x, y = int(rng.integers(0, w)), int(rng.integers(0, h))
            d.line([(x, y), (x - lang // 5, y + lang)], fill=(195, 200, 215), width=2)
        o = o.filter(ImageFilter.GaussianBlur(0.6 + 1.2 * staerke))
        return Image.eval(o, lambda p: int(p * (1 - 0.35 * staerke)))
    if art == "Dunkelheit":
        a = np.asarray(img).astype(np.float32) * (1 - 0.93 * staerke)
        a += rng.normal(0, 18 * staerke, a.shape)          # Kamera-Rauschen wie bei echten Nachtbildern
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    return img

# ---------- Aufkleber ----------
def aufkleber_ort(img, ziel, anteil):
    x1, y1, x2, y2 = ziel["xyxy"]
    g = int(max(12, anteil * (y2 - y1)))
    g = min(g, img.width, img.height)
    cx = int((x1 + x2) / 2); cy = int(y1 + 0.38 * (y2 - y1))
    px = min(max(0, cx - g // 2), img.width - g); py = min(max(0, cy - g // 2), img.height - g)
    return px, py, g

def zufalls_aufkleber(img, ziel, anteil, rng):
    px, py, g = aufkleber_ort(img, ziel, anteil)
    muster = Image.fromarray(rng.integers(0, 256, (g, g, 3), dtype=np.uint8))
    o = img.copy(); o.paste(muster, (px, py)); return o, muster

def berechne_aufkleber(img, ziel, anteil, schritte=40, fortschritt=None):
    """Gezielter Angriff (white-box): In jedem Schritt wird berechnet, in welche Richtung jedes Pixel
    des Musters geändert werden muss, damit die Konfidenz für 'Person' an dieser Stelle sinkt."""
    px, py, g = aufkleber_ort(img, ziel, anteil)
    net = MODELL.model.eval()
    dev = next(net.parameters()).device
    dtype = next(net.parameters()).dtype
    for p in net.parameters(): p.requires_grad_(False)
    x0 = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().div(255).unsqueeze(0).to(dev)
    x1, y1, x2, y2 = ziel["xyxy"]
    patch = torch.rand(1, 3, g, g, device=dev, requires_grad=True)
    for i in range(schritte):
        x = x0.clone(); x[:, :, py:py + g, px:px + g] = patch
        out = net(x.to(dtype)); out = out[0] if isinstance(out, (list, tuple)) else out
        bx, sc = out[0, :4].float(), out[0, 4].float()          # Zeile 4 = Klasse 'person'
        drin = (bx[0] > x1) & (bx[0] < x2) & (bx[1] > y1) & (bx[1] < y2)
        wert = sc[drin].max() if bool(drin.any()) else sc.max()
        wert.backward()
        with torch.no_grad():
            patch -= 0.03 * patch.grad.sign(); patch.clamp_(0, 1); patch.grad.zero_()
        if fortschritt is not None: fortschritt.value = i + 1
    muster = Image.fromarray((patch.detach()[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8))
    o = img.copy(); o.paste(muster, (px, py)); return o, muster

# ---------- Start ----------
neues_bild(Image.open(ASSETS / "bus.jpg"), "Beispielbild (Straßenszene)")
erkenne_alles(S["bild"]); _cache.clear(); erkenne_alles(S["bild"])   # Aufwärmen, damit die Zeitmessung fair ist
display(karte("<b>Bereit.</b> Das Modell ist geladen. Weiter mit <b>Teil 1</b>.", "gut"))


import math
import matplotlib.pyplot as plt
ARTEN = ["Nebel", "Regen", "Dunkelheit"]
WERTE = [round(0.05 * i, 2) for i in range(1, 20)]

def referenz(g):
    """Personen, die bei klarer Sicht mit Schwellenwert g erkannt werden: die Referenz für alle Testfälle."""
    boxen, _d = erkenne_alles(S["bild"])
    return personen(boxen, g)

def testfall(rng):
    """Ein zufälliger Testfall: Sichtbedingung, Stärke und Kompressionsqualität der Kamera."""
    art = str(rng.choice(ARTEN)); staerke = float(rng.uniform(0, 1)); qualitaet = int(rng.integers(20, 96))
    bild = wetter(S["bild"], art, staerke)
    b = io.BytesIO(); bild.save(b, "JPEG", quality=qualitaet)
    return Image.open(b).convert("RGB"), dict(art=art, staerke=staerke, qualitaet=qualitaet)

def bestanden(bild, ziele, g):
    boxen, _d = erkenne_alles(bild)
    return all(bester_treffer(boxen, z) >= g for z in ziele)

def obere_schranke(n, niveau=0.95):
    """Größte Fehlerrate, die bei 0 Fehlern in n Tests mit dem gegebenen Niveau noch nicht ausgeschlossen ist."""
    return 1 - (1 - niveau) ** (1 / n)

def noetige_tests(p, niveau=0.95):
    """Anzahl fehlerfreier Tests, um eine Fehlerrate über p mit dem gegebenen Niveau auszuschließen."""
    return math.ceil(math.log(1 - niveau) / math.log(1 - p))

---
## Teil 1 · Zufällige Testfälle

Jeder **Testfall** ist eine zufällige Variante des Bildes: Nebel, Regen oder Dunkelheit in zufälliger Stärke,
dazu eine zufällige Bildqualität, wie sie bei der Kompression von Kamerabildern entsteht.
Ein Testfall gilt als **bestanden**, wenn alle Personen, die bei klarer Sicht erkannt werden, auch in der Variante erkannt werden.

Eine Menge zufällig ausgewählter Testfälle nennt man **Stichprobe**. Der Anteil nicht bestandener Testfälle ist die beobachtete **Fehlerrate**.

In [ ]:
# @title Teil 1 · Zufällige Testfälle
v1_out = W.Output()
v1_n = W.IntSlider(value=60, min=20, max=200, step=20, description="Testfälle", continuous_update=False,
                   style={"description_width": "90px"}, layout=W.Layout(width="380px"))
v1_g = W.SelectionSlider(options=[(zahl(v), v) for v in WERTE], value=STANDARD, description="Schwellenwert",
                         continuous_update=False, style={"description_width": "110px"}, layout=W.Layout(width="420px"))
v1_los = knopf("Tests ausführen", breite="170px")
v1_neu = knopf("Mit neuem Zufall wiederholen", stil="", breite="250px")
v1_balken = W.IntProgress(value=0, min=0, max=60, description="Fortschritt:", style={"description_width": "90px"},
                          layout=W.Layout(width="380px", visibility="hidden"))
_v1 = {"laeufe": []}

def v1_ausfuehren(seed):
    g = v1_g.value; ziele = referenz(g); n = v1_n.value
    with v1_out:
        v1_out.clear_output(wait=True)
        if not ziele:
            display(karte("Bei klarer Sicht wird keine Person erkannt. Bitte einen niedrigeren Schwellenwert wählen.", "achtung")); return
    rng = np.random.default_rng(seed)
    v1_balken.max = n; v1_balken.value = 0; v1_balken.layout.visibility = "visible"
    faelle = []
    for i in range(n):
        bild, info = testfall(rng)
        info["ok"] = bestanden(bild, ziele, g); info["bild"] = bild if not info["ok"] else None
        faelle.append(info); v1_balken.value = i + 1
    v1_balken.layout.visibility = "hidden"
    fehler = sum(not f["ok"] for f in faelle)
    _v1["laeufe"].append((n, fehler))
    with v1_out:
        v1_out.clear_output(wait=True)
        display(kacheln([("Testfälle", n, "#555"), ("Nicht bestanden", fehler, "#b03a2e" if fehler else "#2e7d4f"),
                         ("Fehlerrate", zahl(fehler / n), "#555")]))
        fig, ax = plt.subplots(figsize=(7, 2.6))
        for k, art in enumerate(ARTEN):
            xs = [f["staerke"] * 100 for f in faelle if f["art"] == art]
            fs = [f["ok"] for f in faelle if f["art"] == art]
            ax.scatter(xs, [k] * len(xs), c=["#2e7d4f" if ok else "#b03a2e" for ok in fs], s=40, alpha=0.8)
        ax.set_yticks(range(len(ARTEN))); ax.set_yticklabels(ARTEN); ax.set_xlabel("Stärke der Sichteinschränkung (%)")
        ax.set_title("Grün: bestanden · Rot: nicht bestanden", fontsize=10); ax.set_xlim(-3, 103)
        plt.tight_layout(); plt.show()
        if len(_v1["laeufe"]) > 1:
            raten = ", ".join(zahl(f / n_) for n_, f in _v1["laeufe"])
            display(karte(f"Beobachtete Fehlerraten aller bisherigen Läufe: {raten}. "
                          "Jeder Lauf ist eine neue Stichprobe, die beobachtete Fehlerrate kann sich dabei ändern. "
                          "Bei kleinen Stichproben, etwa 20 Testfällen, schwankt sie stärker als bei 200.", "info"))
        display(karte("Die Fehlerrate gilt nur für diese Stichprobe: dieses Bild, diese Arten von Störungen, diese Zufallsauswahl. "
                      "Sie ist eine Schätzung, kein exakter Wert.", "frage"))

v1_los.on_click(lambda _: sicher(v1_out)(v1_ausfuehren)(1))
v1_neu.on_click(lambda _: sicher(v1_out)(v1_ausfuehren)(int(time.time())))
display(W.HBox([v1_n, v1_g]), W.HBox([v1_los, v1_neu]), v1_balken, v1_out)

---
## Teil 2 · Was null Fehler bedeuten

Angenommen, ein System besteht alle *n* Testfälle. Ist es dann fehlerfrei? Nein.
Es könnte trotzdem selten versagen, nur eben nicht in dieser Stichprobe.

Die **Dreierregel** (*Rule of Three*) gibt eine einfache Abschätzung: Treten in *n* unabhängigen Tests keine Fehler auf,
kann die tatsächliche Fehlerrate mit 95 % Konfidenzniveau noch bis etwa **3 / n** betragen (Hanley und Lippman-Hand, 1983).
Genauer ist diese **obere Schranke** 1 − 0,05<sup>1/n</sup>.

In [ ]:
# @title Teil 2 · Dreierregel
v2_out = W.Output()
v2_ziel = W.Dropdown(options=[("1 Fehler in 100", 1e-2), ("1 Fehler in 1.000", 1e-3), ("1 Fehler in 100.000", 1e-5),
                              ("1 Fehler in 10.000.000", 1e-7)], value=1e-3, description="Ziel:",
                     style={"description_width": "50px"}, layout=W.Layout(width="300px"))

def tausender(x): return f"{x:,}".replace(",", ".")

def v2_zeigen(*_):
    def pz(x): return f"{x * 100:.2g} %".replace(".", ",")
    zeilen = "".join(f"<tr><td style='padding:2px 12px'>{tausender(n)}</td><td style='padding:2px 12px'>{pz(obere_schranke(n))}</td>"
                     f"<td style='padding:2px 12px'>{pz(3 / n)}</td><td style='padding:2px 12px'>1 Fehler in {tausender(round(1 / obere_schranke(n)))}</td></tr>"
                     for n in (10, 100, 1000, 100000))
    p = v2_ziel.value; n = noetige_tests(p)
    with v2_out:
        v2_out.clear_output(wait=True)
        display(W.HTML("<table style='font-family:sans-serif;font-size:14px;border-collapse:collapse'>"
                       "<tr><th style='padding:2px 12px'>Fehlerfreie Tests</th><th style='padding:2px 12px'>Obere Schranke (exakt)</th>"
                       "<th style='padding:2px 12px'>Dreierregel 3/n</th><th style='padding:2px 12px'>Entspricht etwa</th></tr>" + zeilen + "</table>"))
        display(karte(f"Um eine Fehlerrate über <b>{v2_ziel.label}</b> mit 95 % Konfidenzniveau auszuschließen, "
                      f"sind etwa <b>{tausender(n)} fehlerfreie, unabhängige Tests</b> nötig.", "frage"))
        display(karte("Für sehr seltene Fehler werden reine Zufallstests schnell unpraktikabel. Außerdem setzt die Regel voraus, "
                      "dass die Tests unabhängig sind und die späteren Einsatzbedingungen gut abbilden. Beides ist bei Zufallstests nicht garantiert.", "info"))

v2_ziel.observe(lambda c: sicher(v2_out)(v2_zeigen)(), names="value")
display(v2_ziel, v2_out); v2_zeigen()

---
## Teil 3 · Systematisch suchen

Statt zufällig zu testen, kann man gezielt nach der Stelle suchen, an der das System versagt.
Hier wird für jede erkannte Person und jede Sichtbedingung die Stärke in Schritten von 5 % erhöht,
bis die Person nicht mehr erkannt wird. Diese Stärke ist die **kritische Stärke**, der Testfall an dieser Stelle ein **Grenzfall** (*Edge Case*).

Dieses Vorgehen ist ein einfaches Beispiel für **szenariobasiertes Testen** (*Scenario-based Testing*):
Man beschreibt Szenarien mit ihren Parametern, hier Art und Stärke der Sichteinschränkung, und untersucht sie systematisch.

In [ ]:
# @title Teil 3 · Systematisch suchen
v3_out = W.Output()
v3_g = W.SelectionSlider(options=[(zahl(v), v) for v in WERTE], value=STANDARD, description="Schwellenwert",
                         continuous_update=False, style={"description_width": "110px"}, layout=W.Layout(width="420px"))
v3_los = knopf("Grenzfälle suchen", breite="190px")
v3_balken = W.IntProgress(value=0, min=0, max=1, description="Fortschritt:", style={"description_width": "90px"},
                          layout=W.Layout(width="380px", visibility="hidden"))
STUFEN = [round(0.05 * i, 2) for i in range(1, 21)]

def kritische_staerke(ziel, art, g):
    """Kleinste Stärke (Raster 5 %), bei der die Person nicht mehr erkannt wird; None, wenn sie bis 100 % erkannt bleibt."""
    for s in STUFEN:
        boxen, _d = erkenne_alles(wetter(S["bild"], art, s))
        v3_balken.value += 1
        if bester_treffer(boxen, ziel) < g:
            return s
    return None

@sicher(v3_out)
def v3_klick(_):
    g = v3_g.value; ziele = sorted(referenz(g), key=lambda b: b["xyxy"][0])
    if not ziele:
        with v3_out:
            v3_out.clear_output(); display(karte("Bei klarer Sicht wird keine Person erkannt. Bitte einen niedrigeren Schwellenwert wählen.", "achtung"))
        return
    v3_balken.max = len(ziele) * len(ARTEN) * len(STUFEN); v3_balken.value = 0; v3_balken.layout.visibility = "visible"
    tabelle = {(i, art): kritische_staerke(z, art, g) for i, z in enumerate(ziele) for art in ARTEN}
    geprueft = v3_balken.value
    v3_balken.layout.visibility = "hidden"
    def zelle(s): return "über 100 %" if s is None else f"{round(s * 100)} %"
    kopf = "".join(f"<th style='padding:2px 12px'>{a}</th>" for a in ARTEN)
    zeilen = "".join(f"<tr><td style='padding:2px 12px'>Person {i + 1}</td>" +
                     "".join(f"<td style='padding:2px 12px'>{zelle(tabelle[(i, a)])}</td>" for a in ARTEN) + "</tr>"
                     for i in range(len(ziele)))
    gefunden = [(s, i, a) for (i, a), s in tabelle.items() if s is not None]
    with v3_out:
        v3_out.clear_output(wait=True)
        display(zeige(zeichne(S["bild"], erkenne_alles(S["bild"])[0], g, nummern=ziele, nur_menschen=True), "380px"))
        display(W.HTML("<table style='font-family:sans-serif;font-size:14px;border-collapse:collapse'>"
                       f"<tr><th style='padding:2px 12px'>Kritische Stärke</th>{kopf}</tr>{zeilen}</table>"))
        if gefunden:
            s, i, a = min(gefunden)
            grenzfall = wetter(S["bild"], a, s)
            display(nebeneinander((f"{a} {round(max(s - 0.05, 0) * 100)} %: Person {i + 1} erkannt", zeichne(wetter(S["bild"], a, max(s - 0.05, 0)), erkenne_alles(wetter(S["bild"], a, max(s - 0.05, 0)))[0], g, nur_menschen=True)),
                                  (f"{a} {round(s * 100)} %: Person {i + 1} nicht erkannt", zeichne(grenzfall, erkenne_alles(grenzfall)[0], g, nur_menschen=True))))
        display(karte(f"Nach {geprueft} gezielten Prüfungen ist die kritische Stärke für jede Person und jede Bedingung auf 5 % genau bekannt. "
                      "Zufällige Tests zeigen dagegen nur, ob ein einzelner Testfall bestanden wurde, nicht, ab welcher Stärke das Modell versagt.<br><br>"
                      "Offen bleibt eine Frage, die keine Rechnung beantwortet: <b>Welche Szenarien und Parameter gehören überhaupt in den Test?</b> "
                      "Diese <b>Abdeckung</b> (<i>Coverage</i>) festzulegen und zu begründen ist eine Entscheidung von Menschen.", "frage"))

v3_los.on_click(v3_klick)
display(W.HBox([v3_g, v3_los]), v3_balken, v3_out)

---
## Abschluss

- Zufällige Tests liefern eine **Stichprobe**. Die beobachtete Fehlerrate ist eine Schätzung und schwankt mit dem Zufall.
- **Null Fehler beweisen keine Sicherheit.** Nach der Dreierregel bleibt eine Fehlerrate bis etwa 3 / n möglich.
- **Systematische Suche** findet Grenzfälle gezielt und zeigt, ab welcher Stärke einer Bedingung das System versagt.
- Welche Szenarien getestet werden, legen **Menschen** fest und müssen es begründen.

In der Praxis werden deshalb verschiedene Verfahren kombiniert: Simulation, Tests auf Testgeländen, Fahrten im realen Verkehr
und ein **Sicherheitsnachweis** (*Safety Case*), der begründet, warum das System ausreichend sicher ist.
Sicherheit entsteht also nicht durch Zufall, sondern durch nachvollziehbar begründete Entscheidungen.

### Fachbegriffe

| Begriff | Englisch | Bedeutung |
|---|---|---|
| Testfall | Test Case | Eine konkrete Situation, in der das System geprüft wird. |
| Stichprobe | Sample | Zufällig ausgewählte Menge von Testfällen. |
| Fehlerrate | Failure Rate | Anteil der Testfälle, in denen das System versagt. |
| Konfidenzniveau | Confidence Level | Wie sicher eine statistische Aussage ist, hier 95 %. Nicht zu verwechseln mit der Konfidenz eines Modells. |
| Obere Schranke | Upper Bound | Größter Wert, der statistisch noch nicht ausgeschlossen ist. |
| Dreierregel | Rule of Three | Bei 0 Fehlern in n Tests liegt die obere Schranke der Fehlerrate bei etwa 3 / n (95 %). |
| Grenzfall | Edge Case | Situation, in der ein System gerade noch funktioniert oder gerade versagt. |
| Szenariobasiertes Testen | Scenario-based Testing | Systematische Prüfung beschriebener Szenarien und ihrer Parameter. |
| Abdeckung | Coverage | Wie vollständig die Tests die relevanten Situationen erfassen. |
| Sicherheitsnachweis | Safety Case | Begründete Argumentation, warum ein System ausreichend sicher ist. |

### Weiterführend (Kursmaterial, englisch)

- [SOTIF Scenario Analysis](https://colab.research.google.com/github/milinpatel07/campustage-2026-sicher-durch-zufall/blob/main/04_SOTIF/notebooks/02_scenario_analysis.ipynb)
- [Simulation-Based SOTIF Validation](https://colab.research.google.com/github/milinpatel07/campustage-2026-sicher-durch-zufall/blob/main/04_SOTIF/notebooks/04_simulation_sotif_validation.ipynb)
- [Safety Validation and Testing for AV Perception](https://colab.research.google.com/github/milinpatel07/campustage-2026-sicher-durch-zufall/blob/main/06_AI_Safety/notebooks/05_safety_validation_testing.ipynb)

---
Milin Patel · Hochschule Kempten (IFM) · MIT License. Beispielbild: Ultralytics (AGPL-3.0).